# Review Text Analysis: NLP & Generative AI Pipeline

**Objective:** Preprocess and analyze unstructured text (RAWG game reviews) using NLP and generative AI to extract insights relevant to player churn prediction.

**Pipeline:**
1. Load & normalize RAWG reviews into a unified schema (extensible to Steam later)
2. Preprocess text (strip HTML, clean noise)
3. Sentiment analysis using VADER
4. Keyword extraction from negative reviews
5. GPT-generated summaries of churn-signal complaints
6. Export sentiment features for the churn model


## 0. Setup
Install NLP dependencies if not already present (run once).

In [ ]:
# Uncomment and run once to install NLP extras
# !uv pip install transformers torch openai matplotlib wordcloud

In [ ]:
import json
import os
import re
import sys
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime
from html.parser import HTMLParser
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from dotenv import load_dotenv
from transformers import pipeline

# Make sure the project src is on the path
sys.path.insert(0, str(Path("../src")))

load_dotenv(dotenv_path=Path("../.env"))

print("Imports OK")

In [ ]:
# --- Config ---
RAW_DIR = Path("../data/raw/rawg")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Games we collected reviews for
GAME_SLUGS = ["dota-2", "chess", "league-of-legends"]

# RAWG rating label → normalised 0-1 score
RAWG_RATING_MAP = {
    "exceptional": 1.00,
    "recommended": 0.75,
    "meh":         0.25,
    "skip":        0.00,
}

print(f"RAW_DIR exists: {RAW_DIR.exists()}")

## 1. Unified Review Schema

We define a single `GameReview` dataclass that both RAWG and Steam (future) normalize into.
This keeps the NLP pipeline source-agnostic.

| Field | RAWG | Steam (future) |
|---|---|---|
| `text` | review body | review body |
| `rating` | 0–1 (exceptional/recommended/meh/skip) | 0 or 1 (thumbs) |
| `hours_played` | `None` | hours at time of review |
| `helpful_votes` | `None` | community helpful count |


In [ ]:
@dataclass
class GameReview:
    source: str            # "rawg" | "steam"
    game_slug: str
    review_id: str
    text: str
    rating: float          # normalised 0.0 – 1.0
    timestamp: datetime
    username: str = ""
    # Steam-specific (None for RAWG)
    hours_played: float | None = None
    helpful_votes: int | None = None

print("GameReview schema defined")

## 2. Load & Normalize RAWG Reviews

We load the JSON files saved by `RawgCollector.collect()` and normalize each review into `GameReview`.
If files don't exist yet, we fall back to sample data so the rest of the notebook can run.

In [ ]:
SAMPLE_RAWG_REVIEWS = [
    # dota-2 samples
    {"id": 1, "game_slug": "dota-2", "text": "<p>Amazing depth, but the learning curve is brutal. Toxic community made me quit after 200 hours.</p>", "rating": {"title": "recommended"}, "created": "2024-01-15T10:00:00Z", "user": {"username": "player1"}},
    {"id": 2, "game_slug": "dota-2", "text": "<p>One of the best strategy games ever made. Endless replay value.</p>", "rating": {"title": "exceptional"}, "created": "2024-02-01T08:00:00Z", "user": {"username": "player2"}},
    {"id": 3, "game_slug": "dota-2", "text": "<p>Game is completely unbalanced. Developers don't listen to feedback. Gave up after 3 months.</p>", "rating": {"title": "skip"}, "created": "2024-02-10T12:00:00Z", "user": {"username": "player3"}},
    {"id": 4, "game_slug": "dota-2", "text": "<p>Matchmaking is terrible. Always paired with new players when I've been playing for years.</p>", "rating": {"title": "meh"}, "created": "2024-03-01T09:00:00Z", "user": {"username": "player4"}},
    {"id": 5, "game_slug": "dota-2", "text": "<p>Love this game. The updates keep things fresh. Best MOBA out there.</p>", "rating": {"title": "exceptional"}, "created": "2024-03-05T14:00:00Z", "user": {"username": "player5"}},
    # chess samples
    {"id": 6, "game_slug": "chess", "text": "<p>Perfect for learning chess. Clean interface and good puzzles.</p>", "rating": {"title": "exceptional"}, "created": "2024-01-20T11:00:00Z", "user": {"username": "player6"}},
    {"id": 7, "game_slug": "chess", "text": "<p>Too many bots in lower ranks. Hard to find real opponents. Gets boring fast.</p>", "rating": {"title": "meh"}, "created": "2024-02-05T16:00:00Z", "user": {"username": "player7"}},
    {"id": 8, "game_slug": "chess", "text": "<p>Stopped playing after the rating system changed. Feels unfair now.</p>", "rating": {"title": "skip"}, "created": "2024-02-20T10:00:00Z", "user": {"username": "player8"}},
    # league-of-legends samples
    {"id": 9,  "game_slug": "league-of-legends", "text": "<p>Was fun for years but the monetization is out of control. Skins too expensive.</p>", "rating": {"title": "meh"}, "created": "2024-01-10T13:00:00Z", "user": {"username": "player9"}},
    {"id": 10, "game_slug": "league-of-legends", "text": "<p>Incredible game when you find a good team. Community can be rough though.</p>", "rating": {"title": "recommended"}, "created": "2024-03-10T15:00:00Z", "user": {"username": "player10"}},
    {"id": 11, "game_slug": "league-of-legends", "text": "<p>Quit after 5 years. The new champion designs feel pay-to-win.</p>", "rating": {"title": "skip"}, "created": "2024-03-15T09:00:00Z", "user": {"username": "player11"}},
]

def normalize_rawg_review(raw: dict, game_slug: str) -> GameReview:
    """Convert a raw RAWG API review dict into the unified GameReview schema."""
    rating_title = ""
    if isinstance(raw.get("rating"), dict):
        rating_title = raw["rating"].get("title", "")
    elif isinstance(raw.get("rating"), str):
        rating_title = raw["rating"]

    ts_raw = raw.get("created", "2020-01-01T00:00:00Z")
    try:
        ts = datetime.fromisoformat(ts_raw.replace("Z", "+00:00"))
    except ValueError:
        ts = datetime(2020, 1, 1)

    return GameReview(
        source="rawg",
        game_slug=game_slug,
        review_id=str(raw.get("id", "")),
        text=raw.get("text", ""),
        rating=RAWG_RATING_MAP.get(rating_title, 0.5),
        timestamp=ts,
        username=raw.get("user", {}).get("username", "") if isinstance(raw.get("user"), dict) else "",
        hours_played=None,   # RAWG doesn't provide this
        helpful_votes=None,  # RAWG doesn't provide this
    )


reviews: list[GameReview] = []
using_sample = False

for slug in GAME_SLUGS:
    path = RAW_DIR / f"{slug}_reviews.json"
    if path.exists():
        raw_list = json.loads(path.read_text())
        for r in raw_list:
            reviews.append(normalize_rawg_review(r, slug))
        print(f"  Loaded {len(raw_list):>4} reviews from {path.name}")
    else:
        print(f"  {path.name} not found — will use sample data")
        using_sample = True

if using_sample or len(reviews) == 0:
    print("\nUsing built-in sample data for demonstration.")
    reviews = [
        normalize_rawg_review({k: v for k, v in r.items() if k != "game_slug"}, r["game_slug"])
        for r in SAMPLE_RAWG_REVIEWS
    ]

print(f"\nTotal reviews loaded: {len(reviews)}")

## 3. Text Preprocessing

RAWG review bodies can contain HTML tags. We strip them and normalize whitespace before any NLP.

In [ ]:
class _HTMLStripper(HTMLParser):
    def __init__(self):
        super().__init__()
        self._parts: list[str] = []

    def handle_data(self, data: str) -> None:
        self._parts.append(data)

    def get_text(self) -> str:
        return " ".join(self._parts)


def clean_text(raw: str) -> str:
    """Strip HTML tags, collapse whitespace, lowercase."""
    if not raw or not raw.strip():
        return ""
    stripper = _HTMLStripper()
    stripper.feed(raw)
    text = stripper.get_text()
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Apply to all reviews
for r in reviews:
    r.text = clean_text(r.text)

# Spot check
for r in reviews[:3]:
    print(f"[{r.game_slug}] {r.text[:120]}")

## 4. Sentiment Analysis (HuggingFace Transformers)

We use [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest) — a RoBERTa model fine-tuned on ~124M tweets for 3-class sentiment classification (negative / neutral / positive).

Why this model over a rule-based approach like VADER:
- **Understands context** — "the game is good but the community completely ruined it" → negative (a lexicon would see "good" and score this wrong)
- **Handles nuance** — sarcasm, hedging, compound sentences
- **Trained on social text** — same register as game reviews

Each review gets a `label` (negative/neutral/positive) and a `score` (model confidence 0–1).
We derive a `sentiment_compound` in the range **-1 to +1** for easy thresholding:
- positive → `+score`
- negative → `-score`
- neutral  → `0.0`


In [ ]:
# Load model once — downloads on first run (~500MB), cached locally after
sentiment_pipeline = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=512,
)

def hf_compound(label: str, score: float) -> float:
    """Convert HuggingFace label + confidence into a -1..+1 compound value."""
    if label == "positive":
        return round(score, 4)
    elif label == "negative":
        return round(-score, 4)
    return 0.0  # neutral

# Run in batches — much faster than one review at a time
valid_reviews = [r for r in reviews if r.text]
texts = [r.text for r in valid_reviews]

print(f"Scoring {len(texts)} reviews...")
results = sentiment_pipeline(texts, batch_size=16)

records = []
for r, res in zip(valid_reviews, results):
    label = res["label"].lower()
    score = res["score"]
    records.append({
        "source":             r.source,
        "game_slug":          r.game_slug,
        "review_id":          r.review_id,
        "text":               r.text,
        "rawg_rating":        r.rating,
        "hf_label":           label,
        "hf_score":           round(score, 4),
        "sentiment_compound": hf_compound(label, score),
        "sentiment":          label,
        "timestamp":          r.timestamp,
        "username":           r.username,
    })

df = pl.DataFrame(records)
print(df.select(["game_slug", "rawg_rating", "hf_label", "hf_score", "sentiment_compound"]).head(10))

## 5. Sentiment Distribution

In [ ]:
# --- Overall sentiment breakdown ---
overall = (
    df.group_by("sentiment")
    .agg(pl.len().alias("count"))
    .sort("sentiment")
)
print("Overall sentiment distribution:")
print(overall)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: pie chart overall
labels = overall["sentiment"].to_list()
sizes  = overall["count"].to_list()
colors = {"positive": "#4CAF50", "neutral": "#FFC107", "negative": "#F44336"}
axes[0].pie(
    sizes,
    labels=labels,
    autopct="%1.0f%%",
    colors=[colors.get(l, "gray") for l in labels],
    startangle=90,
)
axes[0].set_title("Overall Sentiment (all games)")

# Right: stacked bar per game
sentiment_order = ["positive", "neutral", "negative"]
per_game = (
    df.group_by(["game_slug", "sentiment"])
    .agg(pl.len().alias("count"))
)
slugs = sorted(df["game_slug"].unique().to_list())
bottoms = [0] * len(slugs)
for sent in sentiment_order:
    vals = []
    for slug in slugs:
        row = per_game.filter((pl.col("game_slug") == slug) & (pl.col("sentiment") == sent))
        vals.append(row["count"][0] if len(row) > 0 else 0)
    axes[1].bar(slugs, vals, bottom=bottoms, label=sent, color=colors[sent])
    bottoms = [b + v for b, v in zip(bottoms, vals)]

axes[1].set_title("Sentiment per Game")
axes[1].set_xlabel("Game")
axes[1].set_ylabel("Review Count")
axes[1].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "sentiment_distribution.png", dpi=120)
plt.show()

In [ ]:
# --- Model confidence vs RAWG user rating ---
# If these correlate, the HuggingFace model is tracking real sentiment.
# Mismatches are interesting: a player who rates "skip" but writes neutrally
# may be less likely to churn than one who rates "skip" AND writes angrily.
print("Avg sentiment_compound by RAWG rating label:")
print(
    df.group_by("rawg_rating")
    .agg(pl.col("sentiment_compound").mean().round(3).alias("avg_compound"), pl.len().alias("n"))
    .sort("rawg_rating", descending=True)
)

## 6. Keyword Extraction from Negative Reviews

We extract the most frequent content words from negative reviews to understand *why* players are unhappy — which maps to churn signals.

In [ ]:
STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "is", "it", "its", "i", "my", "me", "this", "that", "with",
    "have", "has", "had", "been", "be", "are", "was", "were", "after",
    "not", "no", "so", "just", "very", "can", "get", "got", "do", "did",
    "game", "games", "play", "playing", "player", "players",  # domain-general
    "than", "more", "much", "really", "too", "even", "still", "when",
    "you", "your", "they", "their", "he", "she", "we", "our", "from",
    "now", "there", "here", "all", "every", "some", "any", "how", "who",
}

def extract_keywords(texts: list[str], top_n: int = 20) -> list[tuple[str, int]]:
    words: list[str] = []
    for text in texts:
        tokens = re.findall(r"\b[a-z]{3,}\b", text.lower())
        words.extend(t for t in tokens if t not in STOPWORDS)
    return Counter(words).most_common(top_n)


neg_df = df.filter(pl.col("sentiment") == "negative")

print(f"Total negative reviews: {len(neg_df)}\n")
for slug in GAME_SLUGS:
    texts = neg_df.filter(pl.col("game_slug") == slug)["text"].to_list()
    if not texts:
        print(f"{slug}: no negative reviews found")
        continue
    kws = extract_keywords(texts, top_n=10)
    print(f"{slug} — top keywords in negative reviews:")
    for word, count in kws:
        print(f"  {word:<20} {count}")
    print()

In [ ]:
# --- Word frequency bar chart for negative reviews (all games combined) ---
all_neg_texts = neg_df["text"].to_list()
top_kws = extract_keywords(all_neg_texts, top_n=15)

if top_kws:
    words, counts = zip(*top_kws)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(words[::-1], counts[::-1], color="#F44336")
    ax.set_xlabel("Frequency")
    ax.set_title("Most Common Words in Negative Reviews (all games)")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "negative_keywords.png", dpi=120)
    plt.show()

## 7. Generative AI: GPT Summary of Negative Reviews

We feed batches of negative reviews to `gpt-4o-mini` and ask it to:
1. Identify the top recurring complaints
2. Flag churn-predictive signals (toxicity, balance issues, pay-to-win, matchmaking, etc.)
3. Produce a short human-readable summary per game

Requires `OPENAI_API_KEY` in `.env`.

In [ ]:
openai_key = os.getenv("OPENAI_API_KEY", "")

def summarize_negative_reviews(game_slug: str, neg_texts: list[str], max_reviews: int = 30) -> str:
    """Use GPT-4o-mini to summarize churn signals from negative reviews."""
    sample = neg_texts[:max_reviews]
    reviews_block = "\n".join(f"- {t}" for t in sample)
    prompt = (
        f'You are analyzing player reviews for the game "{game_slug}".\n'
        f'Below are {len(sample)} negative reviews.\n\n'
        "Identify:\n"
        "1. The top 3 recurring complaints (be specific, use examples from the text)\n"
        "2. Churn indicators — reasons players say they stopped or are likely to stop playing\n"
        "3. A 2-sentence overall summary of what is driving negative sentiment\n\n"
        f"Reviews:\n{reviews_block}"
    )
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=400,
    )
    return response.choices[0].message.content


gpt_summaries: dict[str, str] = {}

if not openai_key:
    print("OPENAI_API_KEY not set — skipping GPT summaries.")
    print("Add it to .env and re-run this cell.")
else:
    for slug in GAME_SLUGS:
        texts = neg_df.filter(pl.col("game_slug") == slug)["text"].to_list()
        if not texts:
            print(f"{slug}: no negative reviews to summarize")
            continue
        print(f"Summarizing {len(texts)} negative reviews for {slug}...")
        summary = summarize_negative_reviews(slug, texts)
        gpt_summaries[slug] = summary
        print(f"\n=== {slug} ===")
        print(summary)
        print()

## 8. Export Sentiment Features for Churn Model

We aggregate per-game sentiment metrics into a feature table. These will eventually feed into `src/game_churn/features/text_features.py` alongside the behavioral features in `engineer.py`.

When Steam reviews arrive, `hours_at_review` will also appear here as a direct churn predictor.

In [ ]:
sentiment_features = (
    df.group_by("game_slug")
    .agg(
        pl.len().alias("review_count"),
        pl.col("sentiment_compound").mean().round(4).alias("avg_sentiment"),
        pl.col("sentiment_compound").std().round(4).alias("std_sentiment"),
        (pl.col("sentiment") == "positive").sum().alias("positive_count"),
        (pl.col("sentiment") == "neutral").sum().alias("neutral_count"),
        (pl.col("sentiment") == "negative").sum().alias("negative_count"),
        pl.col("rawg_rating").mean().round(4).alias("avg_rawg_rating"),
    )
    .with_columns(
        (pl.col("negative_count") / pl.col("review_count")).round(4).alias("churn_risk_ratio"),
    )
    .sort("churn_risk_ratio", descending=True)
)

print(sentiment_features)

# Save full review-level data
df.write_parquet(OUT_DIR / "rawg_review_sentiment.parquet")
sentiment_features.write_parquet(OUT_DIR / "game_sentiment_features.parquet")

print(f"\nSaved to {OUT_DIR}/")

## 9. Summary of Insights

| Metric | What it tells us |
|---|---|
| `avg_sentiment` | Overall player mood for the game |
| `churn_risk_ratio` | % of reviews that are negative — proxy for retention pressure |
| `std_sentiment` | High variance = polarized community (casual vs hardcore split) |
| Top negative keywords | Concrete pain points: toxicity, balance, matchmaking, monetization |
| GPT summary | Human-readable synthesis of why players quit |

**Next steps (Steam integration):**
- Add `steam.py` collector → normalize into same `GameReview` schema
- `hours_played` at review time becomes a direct churn feature: players who write negative reviews after < 2 hours almost never return
- Merge `game_sentiment_features.parquet` into the main feature table in `engineer.py`
